# Finemapping in srGWS data

In [9]:
!pip3 install polars


[notice] A new release of pip is available: 25.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Data 

In [ ]:
%%bash

# gcloud storage cp --billing-project=$GOOGLE_PROJECT *{tsv,parquet} $WORKSPACE_BUCKET/data/sumstats
gcloud storage cp --billing-project=$GOOGLE_PROJECT  $WORKSPACE_BUCKET/data/sumstats/*{tsv,parquet} .

gcloud storage cp --billing-project=$GOOGLE_PROJECT $WORKSPACE_BUCKET/data/srWGS/FADS/acaf_chr11_61000000-63000000.*

## Software

In [ ]:
%%bash

git clone https://github.com/omerwe/polyfun
cd polyfun

conda config --add pkgs_dirs ~/conda/pkgs
conda config --add envs_dirs ~/conda/envs
conda env create -f polyfun.yml

conda init
source ~/.bashrc
conda activate polyfun
cd ..

## SNPVAR

In [ ]:
%%bash

python polyfun/extract_snpvar.py --sumstats mdd2024_afr_hg38_z.parquet --out mdd2024_afr_hg38_snpvar.tsv

## Update map

Update variant IDs in LD reference to match sumstats.

In [3]:
import pandas as pd

In [22]:
bim = pd.read_csv("acaf_chr11_61000000-63000000.bim", sep = "\t", header = None, names = ["CHR", "ID", "CM", "BP", "A1", "A2"])

snpvar = pd.read_parquet("mdd2024_afr_hg38_snpvar.parquet")

In [37]:
snpvar["CPID_A1A2"] = (
    "chr" +
    snpvar["CHR"].astype(str) + ":" +
    snpvar["BP"].astype(str) + ":" +
    snpvar["A1"].astype(str) + ":" +
    snpvar["A2"].astype(str)
    
)
snpvar["CPID_A2A1"] = (
    "chr" +
    snpvar["CHR"].astype(str) + ":" +
    snpvar["BP"].astype(str) + ":" +
    snpvar["A2"].astype(str) + ":" +
    snpvar["A1"].astype(str)
    
)

In [39]:
bim_rsid_a1a2 = bim.merge(snpvar[["SNP", "CPID_A1A2"]], how = "left", left_on = "ID", right_on = "CPID_A1A2")
bim_rsid_a2a1 = bim.merge(snpvar[["SNP", "CPID_A2A1"]], how = "left", left_on = "ID", right_on = "CPID_A2A1")

In [50]:
map1 = bim_rsid_a1a2[bim_rsid_a1a2["SNP"].notna()][["ID", "SNP"]]
map2 = bim_rsid_a2a1[bim_rsid_a2a1["SNP"].notna()][["ID", "SNP"]]

In [51]:
map2

,ID,SNP
21,chr11:61000492:C:T,rs12791045
23,chr11:61000546:T:A,rs12797044
24,chr11:61000572:G:A,rs2905517
56,chr11:61001264:C:T,rs117030061
58,chr11:61001373:T:A,rs141099168
...,...,...
77363,chr11:62997317:C:T,rs3793960
77382,chr11:62998294:G:A,rs114687557
77392,chr11:62998731:C:T,rs72933455
77393,chr11:62998762:T:A,rs73492018


In [55]:
maps = pd.concat([map1, map2]).drop_duplicates(subset = ["ID"], keep = "first")

In [59]:
maps.to_csv("cpid_rsid.map", header = None, index = None, sep = " ")

## Sample QC

In [77]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness.tsv .
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv .


Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/relatedness/relatedness.tsv...
/ [1 files][  1.0 MiB/  1.0 MiB]                                                
Operation completed over 1 objects/1.0 MiB.                                      
Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv...
/ [1 files][538.5 KiB/538.5 KiB]                                                
Operation completed over 1 objects/538.5 KiB.                                    


In [10]:
relatedness = pd.read_csv("relatedness.tsv", sep = "\t")
flagged_samples = pd.read_csv("flagged_samples.tsv", sep = "\t")

In [11]:
relatedness_i = relatedness[["i.s"]].rename(columns = {"i.s" : "IID"})
relatedness_j = relatedness[["i.s"]].rename(columns = {"i.s" : "IID"})
flagged = flagged_samples[["s"]].rename(columns = {"s" : "IID"})

removed = (
    pd.concat([relatedness_i, relatedness_j, flagged])
    .drop_duplicates(subset = ["IID"])
    .assign(**{"#FID": 0})
    [["#FID", "IID"]]
)

removed.to_csv("flagged.ids", sep = "\t", index = None)

## Genetic clusters

In [61]:
!gsutil -u $GOOGLE_PROJECT cp gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv .

Copying gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv...
/ [1 files][163.4 MiB/163.4 MiB]                                                
Operation completed over 1 objects/163.4 MiB.                                    


In [2]:
ancestry_pred = pd.read_csv("ancestry_preds.tsv", sep = "\t")

In [3]:
(ancestry_pred
       .groupby('ancestry_pred', dropna=False)
       .size()
       .reset_index(name='n'))

,ancestry_pred,n
0,afr,84148
1,amr,79106
2,eas,10099
3,eur,234353
4,mid,1545
5,sas,5579


Randomly select up to 10,000 participants as a reference panel.

In [12]:
clust = (
    ancestry_pred
    [~ancestry_pred["research_id"].isin(removed["IID"])]
    .rename(columns = {'research_id': "IID"})
    .assign(**{"#FID": 0})
    .sample(frac = 1, random_state = 46779)
    .groupby("ancestry_pred", group_keys = False)
    .head(10000)
    .reset_index(drop = True)
    [["#FID", "IID", "ancestry_pred"]]
)

clust.to_csv("clusters.ids", sep = "\t", index = None)

In [13]:
(clust
       .groupby('ancestry_pred', dropna=False)
       .size()
       .reset_index(name='n')
)

,ancestry_pred,n
0,afr,10000
1,amr,10000
2,eas,9615
3,eur,10000
4,mid,1449
5,sas,5299


In [14]:
%%bash

for cluster in afr amr eas eur sas; do
    plink2 \
    --bfile acaf_chr11_61000000-63000000 \
    --make-bed \
    --update-name cpid_rsid.map \
    --keep-col-match clusters.ids ${cluster} \
    --keep-col-match-name ancestry_pred \
    --remove flagged.ids \
    --out acaf_chr11_61000000-63000000_${cluster}
done

PLINK v2.0.0-a.6.12LM 64-bit Intel (20 Apr 2025)   cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to acaf_chr11_61000000-63000000_afr.log.
Options in effect:
  --bfile acaf_chr11_61000000-63000000
  --keep-col-match clusters.ids afr
  --keep-col-match-name ancestry_pred
  --make-bed
  --out acaf_chr11_61000000-63000000_afr
  --remove flagged.ids
  --update-name cpid_rsid.map

Start time: Wed Jan 28 16:02:35 2026
14993 MiB RAM detected, ~13522 available; reserving 7496 MiB for main
workspace.
Using up to 4 compute threads.
414830 samples (0 females, 0 males, 414830 ambiguous; 414830 founders) loaded
from acaf_chr11_61000000-63000000.fam.
77403 variants loaded from acaf_chr11_61000000-63000000.bim.
Note: No phenotype data present.
--update-name: 6052 values updated.
--remove: 381254 samples remaining.
--keep-col-match: 10000 samples remaining.
10000 samples (0 females, 0 males, 10000 ambiguous; 10000 founders) remaining


# Finemapping

In [100]:
!git clone https://github.com/getian107/SuSiEx.git
!chmod a+x SuSiEx/bin_static/SuSiEx

Cloning into 'SuSiEx'...
remote: Enumerating objects: 359, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 359 (delta 52), reused 63 (delta 23), pack-reused 249 (from 1)
Receiving objects: 100% (359/359), 37.42 MiB | 20.24 MiB/s, done.
Resolving deltas: 100% (212/212), done.


In [ ]:
%%bash

sumstats_afr=mdd2024_afr_hg38_susie.tsv
sumstats_amr=mdd2024_his_hg38_susie.tsv
sumstats_eas=mdd2024_eas_hg38_susie.tsv
sumstats_eur=mdd2025_eur_hg38_susie.tsv
sumstats_sas=mdd2024_sas_hg38_susie.tsv

ref_afr=acaf_chr11_61000000-63000000_afr
ref_amr=acaf_chr11_61000000-63000000_amr
ref_eas=acaf_chr11_61000000-63000000_eas
ref_eur=acaf_chr11_61000000-63000000_eur
ref_sas=acaf_chr11_61000000-63000000_sas

ld_afr=ld/acaf_chr11_61000000-63000000_afr
ld_amr=ld/acaf_chr11_61000000-63000000_amr
ld_eas=ld/acaf_chr11_61000000-63000000_eas
ld_eur=ld/acaf_chr11_61000000-63000000_eur
ld_sas=ld/acaf_chr11_61000000-63000000_sas

mkdir -p ld

./SuSiEx/bin_static/SuSiEx \
  --sst_file=$sumstats_afr,$sumstats_eas,$sumstats_amr,$sumstats_sas,$sumstats_eur \
  --n_gwas=70727,58319,16211,14979,1577200 \
  --ref_file=$ref_afr,$ref_eas,$ref_amr,$ref_sas,$ref_eur \
  --ld_file=$ld_afr,$ld_eas,$ld_amr,$ld_sas,$ld_eur \
  --out_dir=. \
  --out_name=mdd_acaf_hg38_chr11_61600000-62000000 \
  --chr=11 \
  --bp=61600000,62000000 \
  --chr_col=1,1,1,1,1 \
  --snp_col=2,2,2,2,2 \
  --bp_col=3,3,3,3,3 \
  --a1_col=4,4,4,4,4 \
  --a2_col=5,5,5,5,5 \
  --eff_col=6,6,6,6,6 \
  --se_col=7,7,7,7,7 \
  --pval_col=8,8,8,8,8 \
  --plink=/usr/bin/plink \
  --keep-ambig=True \
  --maf=0.005 \
  --mult-step=True \
  --pval_thresh=1e-5 \
  --max_iter=500 \
  --tol=1e-4 \
  --threads=30


Software parameters:
--sst_file = mdd2024_afr_hg38_snpvar.tsv,mdd2024_eas_hg38_snpvar.tsv,,mdd2024_sas_hg38_snpvar.tsv,mdd2024_eur_hg38_snpvar.tsv
--ref_file = acaf_chr11_61000000-63000000_afr,acaf_chr11_61000000-63000000_eas,acaf_chr11_61000000-63000000_amr,acaf_chr11_61000000-63000000_sas,acaf_chr11_61000000-63000000_eur
--ld_file = ld/acaf_chr11_61000000-63000000_afr,ld/acaf_chr11_61000000-63000000_eas,ld/acaf_chr11_61000000-63000000_amr,ld/acaf_chr11_61000000-63000000_sas,ld/acaf_chr11_61000000-63000000_eur
--n_gwas = 70727,58319,16211,14979,1577200
--out_dir = .
--out_name = mdd_acaf_hg38_chr11_61000000-63000000
--chr = 11
--bp = 61000000,63000000
--chr_col = 1,1,1,1,1
--snp_col = 2,2,2,2,2
--bp_col = 3,3,3,3,3
--a1_col = 4,4,4,4,4
--a2_col = 5,5,5,5,5
--eff_col = 6,6,6,6,6
--se_col = 7,7,7,7,7
--pval_col = 8,8,8,8,8
--plink = /usr/bin/plink
--keep-ambig = true
--mult-step = true
--precmp = false
--maf = 0.005
--level = 0.95
--min_purity = 0.5
--pval_thresh = 1e-05
--tol = 0.0001

Error: Failed to open ld/acaf_chr11_61000000-63000000_eas_ref.bed.
rm: cannot remove 'ld/acaf_chr11_61000000-63000000_eas.snp': No such file or directory
rm: cannot remove 'ld/acaf_chr11_61000000-63000000_eas_ref.bed': No such file or directory
rm: cannot remove 'ld/acaf_chr11_61000000-63000000_eas_ref.fam': No such file or directory
rm: cannot remove 'ld/acaf_chr11_61000000-63000000_eas_ref.log': No such file or directory
rm: cannot remove 'ld/acaf_chr11_61000000-63000000_eas.log': No such file or directory


... calculate LD matrix: ld/acaf_chr11_61000000-63000000_amr.ld ...
PLINK v1.90b6.22 64-bit (16 Apr 2021)          www.cog-genomics.org/plink/1.9/
(C) 2005-2021 Shaun Purcell, Christopher Chang   GNU General Public License v3
Logging to ld/acaf_chr11_61000000-63000000_amr_ref.log.
Options in effect:
  --bfile acaf_chr11_61000000-63000000_amr
  --chr 11
  --extract ld/acaf_chr11_61000000-63000000_amr.snp
  --keep-allele-order
  --maf 0.005
  --make-bed
  --out ld/acaf_chr11_61000000-63000000_amr_ref

14993 MB RAM detected; reserving 7496 MB for main workspace.
77403 variants loaded from .bim file.
69801 people (0 males, 0 females, 69801 ambiguous) loaded from .fam.
Ambiguous sex IDs written to ld/acaf_chr11_61000000-63000000_amr_ref.nosex .
--extract: 77403 variants remaining.
Using 1 thread (no multithreaded calculations invoked).
Before main variant filters, 69801 founders and 0 nonfounders present.
Calculating allele frequencies... 1011121314151617181920212223242526272829303132333435

In [6]:
cs = pd.read_csv("mdd_acaf_hg38_chr11_61000000-63000000.cs", sep="\t")
snp = pd.read_csv("mdd_acaf_hg38_chr11_61000000-63000000.snp", sep="\t")
summary = pd.read_csv("mdd_acaf_hg38_chr11_61000000-63000000.summary", sep="\t", comment="#")

In [7]:
summary

,CS_ID,CS_LENGTH,CS_PURITY,MAX_PIP_SNP,BP,REF_ALLELE,ALT_ALLELE,REF_FRQ,BETA,SE,-LOG10P,MAX_PIP,POST-HOC_PROB_POP1,POST-HOC_PROB_POP2,POST-HOC_PROB_POP3,POST-HOC_PROB_POP4,POST-HOC_PROB_POP5
0,1,5,0.996141,rs28456,61822009,"A,A,A,A,A","G,G,G,G,G","0.8453,0.3975,0.4564,0.7794,0.7028","-0.0186391,-0.00859807,0.0225219,-0.0148914,-0...","0.00735261,0.00598319,0.0111498,0.0139335,0.00...","1.94909,0.821865,1.36261,0.544877,20.2658",0.398564,0.620173,0.323584,0.618600,0.344088,1.000000
1,2,50,0.706765,rs2957859,61483598,"G,NA,G,NA,G","T,NA,T,NA,T","0.0388,NA,0.0782,NA,0.1273","-0.00933789,NA,-0.0194775,NA,-0.0109132","0.013768,NA,0.0206851,NA,0.00168925","0.303099,NA,0.460438,NA,9.98115",0.072821,0.588716,0.301242,0.341665,0.302463,0.999996


In [8]:
cs

,CS_ID,SNP,BP,REF_ALLELE,ALT_ALLELE,REF_FRQ,BETA,SE,-LOG10P,CS_PIP,OVRL_PIP
0,1,rs28456,61822009,"A,A,A,A,A","G,G,G,G,G","0.8453,0.3975,0.4564,0.7794,0.7028","-0.0186391,-0.00859807,0.0225219,-0.0148914,-0...","0.00735261,0.00598319,0.0111498,0.0139335,0.00...","1.94909,0.821865,1.36261,0.544877,20.2658",0.398564,0.398564
1,1,rs174548,61803876,"C,C,C,C,C","G,G,G,G,G","0.7894,0.3984,0.4452,0.7785,0.7025","-0.0153915,-0.00811976,0.0224246,-0.0144784,-0...","0.00652101,0.0059809,0.0111747,0.0139132,0.001...","1.73849,0.757992,1.34894,0.525711,20.4274",0.349324,0.349324
2,1,rs174560,61814292,"T,T,T,T,T","C,C,C,C,C","0.79,0.3983,0.4452,0.7786,0.704","-0.0149616,-0.00780322,0.0222741,-0.0142633,-0...","0.00652784,0.00598115,0.0111747,0.0139155,0.00...","1.65941,0.716661,1.33505,0.515183,19.9286",0.107933,0.107933
3,1,rs174549,61803910,"G,G,G,G,G","A,A,A,A,A","0.92559,0.3985,0.4738,0.7817,0.7087","-0.0155347,-0.00772335,0.0224972,-0.0147409,-0...","0.0101314,0.00598064,0.0111226,0.0139861,0.001...","0.902417,0.706487,1.36543,0.53477,20.2658",0.073504,0.073504
4,1,rs174544,61800281,"C,C,C,C,C","A,A,A,A,A","0.92578,0.3991,0.4741,0.782,0.7115","-0.0159587,-0.0076422,0.0206248,-0.0129935,-0....","0.0101433,0.00597913,0.0111223,0.013993,0.0012...","0.936886,0.696376,1.19595,0.452088,19.769",0.022428,0.022428
5,2,rs2957859,61483598,"G,NA,G,NA,G","T,NA,T,NA,T","0.0388,NA,0.0782,NA,0.1273","-0.00933789,NA,-0.0194775,NA,-0.0109132","0.013768,NA,0.0206851,NA,0.00168925","0.303099,NA,0.460438,NA,9.98115",0.072821,0.072821
6,2,rs112830700,61366508,"G,NA,G,NA,G","A,NA,A,NA,A","0.9773,NA,0.92489,NA,0.8749","-0.00530825,NA,0.028198,NA,0.0108386","0.0178511,NA,0.021071,NA,0.0017019","0.115663,NA,0.742754,NA,9.71936",0.060030,0.060030
7,2,rs3741265,61397808,"G,G,G,G,G","A,A,A,A,A","0.2387,0.0381,0.121,0.119,0.1416","-0.0132821,0.0139687,-0.00675831,0.00209365,-0...","0.00623718,0.0152951,0.0170292,0.0178436,0.001...","1.47869,0.442378,0.160229,0.0425862,10.2629",0.052872,0.052872
8,2,rs2924443,61466040,"G,G,G,G,G","A,A,A,A,A","0.2395,0.0384,0.1225,0.1228,0.1401","-0.0150596,0.0146165,-0.00560117,0.00501601,-0...","0.00623003,0.0152376,0.016939,0.0176033,0.0016...","1.80583,0.471806,0.130243,0.110314,9.97475",0.045483,0.045483
9,2,rs896832,61437167,"A,A,A,A,A","G,G,G,G,G","0.2252,0.0381,0.1202,0.1193,0.14","-0.0152942,0.0126423,-0.0114298,0.0017203,-0.0...","0.00636522,0.0152951,0.017078,0.0178242,0.0016...","1.78858,0.388821,0.298153,0.0347457,9.8078",0.038666,0.038666


In [9]:
snp

,BP,SNP,PIP(CS1),"LogBF(CS1,Pop1)","LogBF(CS1,Pop2)","LogBF(CS1,Pop3)","LogBF(CS1,Pop4)","LogBF(CS1,Pop5)",PIP(CS2),"LogBF(CS2,Pop1)","LogBF(CS2,Pop2)","LogBF(CS2,Pop3)","LogBF(CS2,Pop4)","LogBF(CS2,Pop5)"
0,61000222,rs2905516,1.890870e-17,-0.659646,-1.395760e-10,-8.833080e-01,-2.753260e-10,-7.737970e-02,4.888940e-10,-0.696585,-1.395760e-10,-1.053750e+00,-2.753260e-10,-5.348240e-01
1,61000351,rs11230558,1.799620e-18,-0.431094,-1.403450e+00,-1.129690e+00,-9.554910e-01,-5.264940e-02,5.649950e-11,-0.202143,-1.393740e+00,-1.200090e+00,-9.938440e-01,-6.532520e-01
2,61000492,rs12791045,2.224120e-19,-1.211350,-4.724360e-01,-1.048000e+00,-1.104930e+00,-2.226470e+00,1.086310e-11,-1.185190,-4.222030e-01,-1.202990e+00,-1.094840e+00,-2.186700e+00
3,61000546,rs12797044,2.180060e-19,-1.213980,-4.762670e-01,-1.037850e+00,-1.105140e+00,-2.249950e+00,1.018620e-11,-1.188200,-4.261810e-01,-1.201590e+00,-1.095180e+00,-2.245120e+00
4,61000572,rs2905517,2.453750e-18,-0.055383,-4.223810e-01,-5.016490e-01,-5.094040e-01,-2.173520e+00,6.934770e-11,-0.017363,-4.677850e-01,-1.148450e+00,-4.965280e-01,-2.108050e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6032,62998397,rs4149183,1.002040e-17,-1.237420,-1.248610e+00,-3.151510e-01,-8.838310e-01,1.429680e+00,5.429310e-10,-1.246550,-1.304100e+00,-8.550390e-01,-8.276870e-01,2.053060e+00
6033,62998731,rs72933455,3.270790e-19,-1.280770,-1.416270e+00,-8.684650e-01,-2.753260e-10,-2.112010e+00,1.579610e-11,-1.287350,-1.412330e+00,-9.876090e-01,-2.753260e-10,-2.030250e+00
6034,62998762,rs73492018,9.945480e-18,-1.061470,-1.395760e-10,-1.201360e+00,-2.753260e-10,-2.824900e-11,5.329010e-10,-1.009960,-1.395760e-10,-1.189000e+00,-2.753260e-10,-2.824900e-11
6035,62998959,rs2276299,2.177900e-17,-0.190431,1.566050e+00,-1.172080e+00,-7.833620e-01,-8.991860e-01,1.290600e-09,-0.017174,1.485620e+00,-1.187090e+00,-7.661650e-01,-8.296320e-01
